# Grid resolution analysis (script)

Script version of the notebook on the documentation website, without the theory and schematics. Set `XFOIL_BIN` in the parameters cell to the path of your XFOIL binary, then run all cells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Data

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

GRID_FILE = "../data/grid_wall.hdf5"
SOL_FILE  = "../data/solution_wall.hdf5"
BLOCK     = "block0"

GAMMA = 1.4        # ratio of specific heats
S     = 110.4      # Sutherland constant [K]
T_REF = 273.15     # reference temperature [K]

def read(group, name):
    # Instantaneous files store u, v, ...; time-averaged files store um, vm, ...
    return group[name] if name in group else group[name + "m"]

with h5py.File(GRID_FILE, "r") as fg, h5py.File(SOL_FILE, "r") as fs:
    x, y, z = (fg[BLOCK][c][:, 0:2, :] for c in "xyz")       # node layers j = 0, 1
    u, v, w, rho, p = (read(fs[BLOCK], n)[:, 0, :] for n in ("u", "v", "w", "rho", "p"))
    ma, re = float(np.squeeze(fs.attrs["Ma"])), float(np.squeeze(fs.attrs["Re"]))

print(f"Grid nodes (k, i): {x[:, 0, :].shape},  wall cells (k, i): {u.shape}")

## Grid spacings at the wall

In [ ]:
def face_centres(a):
    # Node values (k, i) -> face-centre values (k-1, i-1)
    return 0.25*(a[:-1, :-1] + a[1:, :-1] + a[:-1, 1:] + a[1:, 1:])

wall_nodes = np.stack([c[:, 0, :] for c in (x, y, z)])            # (3, nk, ni)
wall_face  = np.stack([face_centres(c[:, 0, :]) for c in (x, y, z)])
top_face   = np.stack([face_centres(c[:, 1, :]) for c in (x, y, z)])
centre     = 0.5*(wall_face + top_face)                            # first cell centres

d  = centre - wall_face
y1 = np.linalg.norm(d, axis=0)                                     # wall distance
n  = d/y1                                                          # unit wall normal

dx = np.linalg.norm(np.diff(wall_nodes, axis=2), axis=0)          # edges along i
dx = 0.5*(dx[:-1, :] + dx[1:, :])
dz = np.linalg.norm(np.diff(wall_nodes, axis=1), axis=0)          # edges along k
dz = 0.5*(dz[:, :-1] + dz[:, 1:])

## Friction velocity and wall units

In [ ]:
def sutherland(T):
    return T**1.5*(1 + S/T_REF)/(T + S/T_REF)

t_inf   = 1.0/(1.0 + 0.5*(GAMMA - 1.0)*ma**2)
u_inf   = ma*np.sqrt(t_inf)
rho_inf = t_inf**(1.0/(GAMMA - 1.0))
re0     = re*sutherland(t_inf)/(rho_inf*u_inf)      # Reynolds number per unit length

T  = GAMMA*p/rho
mu = sutherland(T)
nu = mu/rho

vel  = np.stack([u, v, w])
u_t  = np.linalg.norm(vel - np.sum(vel*n, axis=0)*n, axis=0)   # wall-tangential velocity
tau  = mu/re0*u_t/y1
utau = np.sqrt(tau/rho)

inv_lnu = utau*re0/nu                                # 1/ℓν
metrics = {r"$y^+$": y1*inv_lnu, r"$\Delta x^+$": dx*inv_lnu, r"$\Delta z^+$": dz*inv_lnu}

## Suction and pressure side

In [ ]:
xc = centre[0].mean(axis=0)
i_le = np.argmin(xc)
sides = {"a": np.arange(0, i_le + 1), "b": np.arange(i_le, len(xc))}
if p[:, sides["a"]].mean() < p[:, sides["b"]].mean():
    sides = {"suction": sides["a"], "pressure": sides["b"]}
else:
    sides = {"suction": sides["b"], "pressure": sides["a"]}

def surface_stats(idx):
    idx = idx[np.argsort(xc[idx])]
    return xc[idx], {k: (m.mean(axis=0)[idx], m.min(axis=0)[idx], m.max(axis=0)[idx])
                     for k, m in metrics.items()}

def plot_side(name, color, x_max=0.9):
    xs, stats = surface_stats(sides[name])
    keep = xs <= x_max
    fig, axs = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
    for ax, (label, (mean, lo, hi)) in zip(axs, stats.items()):
        ax.fill_between(xs[keep], lo[keep], hi[keep], color=color, alpha=0.2, lw=0, label="spanwise min–max")
        ax.plot(xs[keep], mean[keep], color=color, lw=1.5, label="spanwise mean")
        ax.set_ylabel(label)
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)
    axs[0].axhline(1.0, color="k", ls="--", lw=1, label="$y^+ = 1$")
    axs[0].legend(loc="upper right", fontsize=8)
    axs[-1].set_xlabel("$x/c$")
    axs[-1].set_xlim(0, x_max)
    fig.suptitle(f"Grid resolution: {name} side", y=0.93)
    plt.show()

## Results

In [ ]:
plot_side("suction", "tab:red")

In [ ]:
plot_side("pressure", "tab:blue")